In [2]:
from typing import Iterable

import pandas as pd
from common.utils import get_project_root
from custom_nlp.tratamento_texto import renomeia_cols

In [3]:
texto_para_remover = "VERSÃO PRELIMINAR (Esta versão será substituída após conclusão da revisão de perfis, conhecimentos, habilidades, atitudes e níveis)"


def agg_f(textos: Iterable[str]) -> str:
    textos_sem_versao_preliminar = set(
        map(lambda texto: texto.replace(texto_para_remover, "").strip(), textos)
    )
    return ". ".join(textos_sem_versao_preliminar)


def agg_textos(
    df: pd.DataFrame, agg_col: str, cols_with_text: list[str]
) -> pd.DataFrame:
    return (
        df.dropna(subset=cols_with_text)
        .groupby(by=agg_col, as_index=False)[cols_with_text]
        .agg(agg_f)
        .rename(columns={agg_col: "codigo"})
    )

In [4]:
project_root_dir = get_project_root("classificador-cbo")

# Carregando os dados
qbq_path = project_root_dir / "data/bronze/OcupacoesCBO.xlsx"
cols_interesse = ["CodCBO", "Ocupação", "Síntese", "PerfilOcupacional"]
df = pd.read_excel(qbq_path, sheet_name="Ocupação", usecols=cols_interesse, dtype=str)

# Pegando os códigos
df["grande_grupo"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:1])
df["subgrupo_principal"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:2])
df["subgrupo"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:3])
df["familia"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:4])

classificacoes = [
    ("grande_grupo", "Grande Grupo"),
    ("subgrupo_principal", "SubGrupo Principal"),
    ("subgrupo", "SubGrupo"),
    ("familia", "Familia"),
]

for classificacao in classificacoes:
    # Obtendo os textos
    textos = renomeia_cols(
        agg_textos(df, classificacao[0], ["Síntese", "PerfilOcupacional"])
    )
    path_csv_titulos = (
        project_root_dir / f"data/bronze/CBO2002 - {classificacao[1]}.csv"
    )

    titulos = renomeia_cols(
        pd.read_csv(path_csv_titulos, dtype=str, encoding="latin1", sep=";")
    )

    textos_e_titulos = pd.merge(left=textos, right=titulos, on="codigo", how="left")

    path_csv_destino = project_root_dir / f"data/silver/{classificacao[0]}.csv"
    textos_e_titulos.to_csv(path_csv_destino, index=False, sep="\t", encoding="utf-8")

In [5]:
df.head(10)

,CodCBO,Ocupação,Síntese,PerfilOcupacional,grande_grupo,subgrupo_principal,subgrupo,familia
0,312105,Técnico de obras civis,"Desenvolve e legaliza projetos de obras civis,...",Realiza levantamentos topográficos e planialti...,3,31,312,3121
1,313105,Eletrotécnico,Elabora estudos e projetos referentes a sistem...,Elabora estudos e projetos referentes a sistem...,3,31,313,3131
2,313110,Eletrotécnico (produção de energia),Elabora estudos e projetos em sistemas de prod...,Elabora estudos e projetos em sistemas de prod...,3,31,313,3131
3,313115,"Eletrotécnico na fabricação, montagem e instal...",Realiza projetos de máquinas e equipamentos el...,Realiza projetos de máquinas e equipamentos el...,3,31,313,3131
4,313120,Técnico de manutenção elétrica,Executa manutenção elétrica preventiva e corre...,Executa manutenção elétrica preventiva e corre...,3,31,313,3131
5,313125,Técnico de manutenção elétrica de máquina,Executa manutenção elétrica - preventiva e cor...,Executa manutenção elétrica preventiva e corre...,3,31,313,3131
6,313130,Técnico eletricista,Elabora estudos e projetos em sistemas elétric...,Elabora estudos e projetos em sistemas elétric...,3,31,313,3131
7,313205,Técnico de manutenção eletrônica,"Instala, conserta e faz manutenção corretiva, ...","Instala equipamentos e aparelhos eletrônicos, ...",3,31,313,3132
8,313210,Técnico de manutenção eletrônica (circuitos de...,"Instala, conserta e faz manutenção corretiva, ...",Conserta e faz manutenção corretiva em circuit...,3,31,313,3132
9,313215,Técnico eletrônico,"Desenvolve projetos eletrônicos, monta e testa...",Conserta e faz manutenção corretiva em aparelh...,3,31,313,3132


In [12]:
import gensim.models

sentences = [texto.split() for texto in df.loc[:, "Síntese"].dropna().values]
model = gensim.models.Word2Vec(sentences=sentences)

In [30]:
eng_eletrico = "Engenheiro Eletricista com formação sólida e experiência em projetos, manutenção e operação de sistemas elétricos de baixa, média e alta tensão. Atuação em setores industriais, comerciais e/ou residenciais, com foco em eficiência energética, automação e segurança elétrica. Conhecimento aprofundado em normas técnicas (como a NBR 5410 e NR-10), elaboração de diagramas elétricos, uso de softwares especializados (AutoCAD, EPLAN, Matlab, etc.) e gestão de equipes multidisciplinares. Perfil analítico, proativo e comprometido com soluções técnicas seguras, sustentáveis e economicamente viáveis."
prof_geografia = "Professor de Geografia com sólida formação acadêmica e mais de [X anos] de experiência no ensino fundamental, médio e/ou superior. Especialista em metodologias ativas de aprendizagem, com foco em tornar o ensino da Geografia mais dinâmico, contextualizado e interdisciplinar. Ampla vivência em sala de aula, desenvolvendo projetos pedagógicos voltados à educação ambiental, geopolítica e cidadania. Excelente capacidade de comunicação, planejamento de aulas e avaliação de desempenho discente. Comprometido com a formação crítica dos alunos e com o uso de recursos tecnológicos para potencializar o aprendizado."
modelista = "Modelista com ampla experiência no desenvolvimento de moldes para confecção de peças do vestuário feminino, masculino e infantil. Domínio em modelagem plana, moulage e interpretação de fichas técnicas, com sólida capacidade de transformar croquis e ideias em peças-piloto de alto padrão. Conhecimento técnico em tecidos, caimentos e encaixes, além de domínio de softwares de modelagem como Audaces, Gerber e Lectra. Perfil detalhista, criativo e comprometido com a qualidade, prazos e processos de produção em ateliês ou confecções industriais."

In [36]:
from custom_nlp.embeddings import load_embedding_model

w2v = load_embedding_model("w2v_familiaCbo_normalizado_s_stopword")

a = w2v.wv.get_mean_vector(df.iloc[6, 3].split())
b = w2v.wv.get_mean_vector(eng_eletrico.split())
c = w2v.wv.get_mean_vector(prof_geografia.split())
d = w2v.wv.get_mean_vector(modelista.split())

w2v.wv.cosine_similarities(vector_1=a, vectors_all=[b, c, d])

array([0.97177094, 0.82019943, 0.8858005 ], dtype=float32)

In [33]:
from gensim.utils import simple_preprocess

simple_preprocess(modelista)

['modelista',
 'com',
 'ampla',
 'experiência',
 'no',
 'desenvolvimento',
 'de',
 'moldes',
 'para',
 'confecção',
 'de',
 'peças',
 'do',
 'vestuário',
 'feminino',
 'masculino',
 'infantil',
 'domínio',
 'em',
 'modelagem',
 'plana',
 'moulage',
 'interpretação',
 'de',
 'fichas',
 'técnicas',
 'com',
 'sólida',
 'capacidade',
 'de',
 'transformar',
 'croquis',
 'ideias',
 'em',
 'peças',
 'piloto',
 'de',
 'alto',
 'padrão',
 'conhecimento',
 'técnico',
 'em',
 'tecidos',
 'caimentos',
 'encaixes',
 'além',
 'de',
 'domínio',
 'de',
 'softwares',
 'de',
 'modelagem',
 'como',
 'audaces',
 'gerber',
 'lectra',
 'perfil',
 'detalhista',
 'criativo',
 'comprometido',
 'com',
 'qualidade',
 'prazos',
 'processos',
 'de',
 'produção',
 'em',
 'ateliês',
 'ou',
 'confecções',
 'industriais']

# A

In [1]:
# =============================================================================
# BIBLIOTECAS E MÓDULOS
# =============================================================================

import numpy as np
import pandas as pd
from common.utils import get_project_root
from custom_nlp.embeddings import CountVect, Embedder, TfidfVect, load_embedding_model
from custom_nlp.tratamento_texto import (
    NormalizationStrategy,
    Preprocessor,
    StopwordsRemovalStrategy,
)

# =============================================================================
# CONSTANTES
# =============================================================================

project_root_dir = get_project_root("classificador-cbo")
treated_data_dir = project_root_dir / "data/silver/cbo_sintese_perfil"
embedding_models_dir = project_root_dir / "models/embedding"
embedding_models_dir.mkdir(exist_ok=True, parents=True)

teste = """**Título:** Contador Financeiro  

**Descrição:**  
Estamos em busca de um contador financeiro altamente qualificado para integrar nossa equipe. O profissional será responsável por gerenciar e analisar as demonstrações financeiras, garantindo a conformidade com as normas contábeis e fiscais.  

**Responsabilidades:**  
- Elaborar e analisar relatórios financeiros detalhados  
- Monitorar e garantir a conformidade fiscal e contábil  
- Gerenciar fluxo de caixa e planejamento financeiro  
- Fornecer suporte contábil para decisões estratégicas  
- Trabalhar em conjunto com outras áreas para garantir a precisão das informações financeiras  

**Requisitos:**  
- Graduação em Contabilidade, Finanças ou área relacionada  
- Experiência prévia na área contábil ou financeira  
- Conhecimento em normas contábeis e legislação fiscal  
- Domínio de ferramentas financeiras e contábeis  
- Habilidades analíticas e atenção aos detalhes"""

# =============================================================================
# FUNÇÕES
# =============================================================================


# =============================================================================
# CONSTANTES
# =============================================================================

count_vect = load_embedding_model("countVect_familiaCbo_normalizado_s_stopword")
tfidf_vect = load_embedding_model("tfidfVect_familiaCbo_normalizado_s_stopword")

In [ ]:
embedder = Embedder(strategy=CountVect, model=count_vect)
embedding_ref_count = embedder.apply(textos_normalizados_s_stopwords)
embedding_texto_count = embedder.apply(teste)

embedder.set_strategy(strategy=TfidfVect, model=tfidf_vect)
embedding_ref_tfidf = embedder.apply(textos_normalizados_s_stopwords)
embedding_texto_tfidf = embedder.apply(teste)

'tfidfvectorizer'

In [ ]:
tfidf_vect = TfidfVectorizer()
preprocessador = Preprocessor()

for csv_file_path in treated_data_dir.rglob("*.csv"):
    dados_referencia = pd.read_csv(csv_file_path, sep="\t")
    textos = dados_referencia.loc[:, "sintese"].values

    preprocessador.set_strategy(strategy=NormalizationStrategy)
    textos_normalizados = np.vectorize(pyfunc=preprocessador.apply)(textos)

    preprocessador.set_strategy(strategy=StopwordsRemovalStrategy)
    textos_normalizados_s_stopwords = np.vectorize(pyfunc=preprocessador.apply)(
        textos_normalizados
    )

    tfidf_vect.fit(raw_documents=textos_normalizados_s_stopwords)

[PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/brf-831771-ds3v2/code/Users/lucas.alecrim/classificador-cbo/data/silver/cbo_sintese_perfil/familia.csv'),
 PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/brf-831771-ds3v2/code/Users/lucas.alecrim/classificador-cbo/data/silver/cbo_sintese_perfil/grande_grupo.csv'),
 PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/brf-831771-ds3v2/code/Users/lucas.alecrim/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo.csv'),
 PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/brf-831771-ds3v2/code/Users/lucas.alecrim/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo_principal.csv')]

In [9]:
import numpy as np
import pandas as pd

df_amostra = pd.read_parquet(
    "../../../banco-talentos/data/silver/amostra_preprocessado.parquet"
)
embedding_ref_tfidf = embedder.apply(textos_normalizados_s_stopwords)

familias_cbo_candidatos = []
for i in range(len(df_amostra)):
    texto_cv_candidato = df_amostra.iloc[i, -1]
    embedding_cv_candidato = embedder.apply(texto_cv_candidato)
    dot_tfidf = np.dot(embedding_ref_tfidf, embedding_cv_candidato.T)
    idx_argmax_tfidf = dot_tfidf.argmax()
    familias_cbo_candidatos.append(df.iloc[idx_argmax_tfidf, -1])

df_amostra["familia_cbo"] = familias_cbo_candidatos

In [17]:
df_amostra.head(10)

,text,processing_datetime,sys_record_update_time,transformed_text,familia_cbo
attachmentId,,,,,
4697678,LEANDRO HIPÓLITO AFONSO MACHADO DADOS: Brasile...,2024-04-04 16:17:49.365102,2024-06-26 08:34:03,leandro hipólito Afonso machar dado brasileiro...,Especialistas em promoção de produtos e vendas
4688899,Ester Rocha da Silva Rua Manoel Messias Da Sil...,2024-04-02 19:14:26.836700,2024-06-26 08:34:03,ester rocha Silva rua Manoel Messias Silva n j...,"Gerentes de comercialização, marketing e comun..."
4662513,José Carlos Decaro Soares De Souza José Gonçal...,2024-03-26 22:30:13.999383,2024-06-26 08:34:03,José Carlos decaro Soares Souza José Gonçalves...,Professores práticos no ensino profissionalizante
4094090,"Gabriela Nicastro \n32 anos, casada\n(11) 9524...",2024-03-26 22:30:13.999383,2024-06-26 08:34:03,gabrielo nicastro ano casado gaby_rtds hotmail...,Especialistas em promoção de produtos e vendas
4422341,Vitória Alves Rodrigues Celular: (151997946630...,2024-03-26 22:30:13.999383,2024-06-26 08:34:03,vitória Alves rodrigues celular email viihr al...,Professores do ensino médio
4330634,Lucas Bernardi Machado Solteiro; 26 anos. Rua ...,2024-03-26 22:30:13.999383,2024-06-26 08:34:03,luca Bernardi machar solteiro ano rua isaio fo...,Engenheiros agrossilvipecuários
2632561,JOSÉ LUCAS RODRIGUES DOS SANTOS 23 anos Soltei...,2024-03-26 22:30:13.999383,2024-06-26 08:34:03,José luco rodrigue santo ano solteiro dar pess...,Profissionais de administração ecônomico-finan...
2464054,"\nLucas José Nunes da Silva\nJiquiá, Recife/P...",2024-03-26 22:30:13.999383,2024-06-26 08:34:03,luco José Nunes Silva jiquiá recife pe fone oi...,Almoxarifes e armazenistas
4036982,Paulo Sérgio Da Silva Lima 1. DADOS PESSOAIS: ...,2024-03-26 22:30:13.999383,2024-06-26 08:34:03,Paulo sérgio Silva lima dar pessoal recife per...,Especialistas em promoção de produtos e vendas


In [22]:
embedder.apply(text=df_amostra.loc[4036982, "text"]).todense()

matrix([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 11001))

In [20]:
from pprint import pprint

pprint(df_amostra.loc[4036982, "text"])

('Paulo Sérgio Da Silva Lima 1. DADOS PESSOAIS: Recife Pernambuco Brasileiro '
 'silvalima1985@gmail com Solteiro Contato: (81) 98899-8244 Nascimento: '
 '25/08/1985 Carro próprio e disponível para viagem. 2 OBJETIVO: Desenvolver '
 'uma experiência profissional ao lado dos meus conhecimentos; contribuindo '
 'sempre para 0 alcance dos objetivos da empresa. 3 FORMAÇÃO ~Ensino superior '
 '(Logística) -Pós Gestão de Pessoas e Mkt 4 QUALIFICAÇÃO PROFISSIONAL Pacote '
 'Office. ~Inglês Básico (Em andamento) -Plano de Marketing -Marketing '
 '~Marketing Aplicado 5 HISTÓRICO PROFISSIONAL Empresa: Vivo SIA '
 'Período:09/05/201 1 á 03/02/2014 Funcão: Consultor de Vendas '
 'Atividades_Desenvolvidas: -Vendas digitais (E-mail; WhatsApp; Link). -Vendas '
 'Consultiva de Serviços. ~Ações Externa (Captação de leads) .Empresa: Fronte '
 'Imóveis Período: 20/02/2016 á 10/12/2017 Função: Corretor de '
 'imóveislcoordenador Atividades_ Desenvolvidas: ~Vendas digitais (E-mail, '
 'WhatsApp; Link) . ~V

In [13]:
import numpy as np

dot_count = np.dot(embedding_ref_count, embedding_texto_count.T)
idx_argmax_count = dot_count.argmax()
print(df.iloc[idx_argmax_count, -1])

dot_tfidf = np.dot(embedding_ref_tfidf, embedding_texto_tfidf.T)
idx_argmax_tfidf = dot_tfidf.argmax()
print(df.iloc[idx_argmax_tfidf, -1])
# a[idx_argmax]

Médicos clínicos
Contadores e afins


In [15]:
df.iloc[116]

codigo                                                            2522
sintese              Realiza perícias contábeis judiciais, extrajud...
perfilocupacional    Realiza perícias contábeis judiciais, extrajud...
titulo                                              Contadores e afins
Name: 116, dtype: object

In [4]:
for a in treated_data_dir.rglob("*.csv"):
    print(a)

/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/grande_grupo.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo_principal.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/familia.csv
